# Models

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_curve, f1_score
import matplotlib.pyplot as plt
import numpy as np

In [2]:
df = pd.read_csv("/Users/jasonc/Desktop/DSC_291/processed_data.csv")

In [3]:
df["FPF_TARGET"].value_counts()

FPF_TARGET
0.0    13397
1.0     2455
Name: count, dtype: int64

In [4]:
# Extract features and labels
X = df.drop(columns=["FPF_TARGET"]).select_dtypes(include=[np.number]).values.astype(np.float32)
y = df["FPF_TARGET"].values.astype(np.float32)

In [5]:
# Stratified train/validation split (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [6]:
# Logistic regression pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

In [7]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('logreg',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [8]:
# Predict probabilities
y_probs = pipeline.predict_proba(X_val)[:, 1]

In [9]:
# Precision-recall curve
prec, rec, thresholds = precision_recall_curve(y_val, y_probs)
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Best threshold: {best_threshold:.4f}")
print(f"Precision at best F1: {prec[best_idx]:.4f}")
print(f"Recall at best F1: {rec[best_idx]:.4f}")
print(f"F1 score: {f1_scores[best_idx]:.4f}")

Best threshold: 0.4946
Precision at best F1: 0.3081
Recall at best F1: 0.7862
F1 score: 0.4427


In [10]:
# Final predictions using tuned threshold
y_pred_custom = (y_probs >= best_threshold).astype(int)

In [11]:
# Evaluate
print("\n=== Validation Report (Custom Threshold) ===")
print(classification_report(y_val, y_pred_custom, digits=4))


=== Validation Report (Custom Threshold) ===
              precision    recall  f1-score   support

         0.0     0.9453    0.6765    0.7886      2680
         1.0     0.3081    0.7862    0.4427       491

    accuracy                         0.6935      3171
   macro avg     0.6267    0.7313    0.6156      3171
weighted avg     0.8466    0.6935    0.7350      3171



In [12]:
# Predict on training data
y_train_probs = pipeline.predict_proba(X_train)[:, 1]
y_train_pred = (y_train_probs >= 0.5).astype(int)

# Evaluate on training set
train_accuracy = accuracy_score(y_train, y_train_pred)
train_report = classification_report(y_train, y_train_pred, output_dict=True)
train_report_df = pd.DataFrame(train_report).transpose()

train_accuracy, train_report_df

(0.700023657440265,
               precision    recall  f1-score       support
 0.0            0.947559  0.682840  0.793709  10717.000000
 1.0            0.314441  0.793788  0.450448   1964.000000
 accuracy       0.700024  0.700024  0.700024      0.700024
 macro avg      0.631000  0.738314  0.622079  12681.000000
 weighted avg   0.849504  0.700024  0.740546  12681.000000)